In [ ]:
# NOTE : 01_Load_Cutout.ipynb - Cell 1 

from IPython.display import HTML, display, clear_output, Javascript

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout,
.cell-output, .cell-output-print, .cell-output-stdout {
    color: white !important;
}
.output * { color: white !important; }
.output .ansi-yellow-fg, .output .ansi-yellow-foreground,
.warning, .Warning, .warnings {
    color: #FFD700 !important;
    text-shadow: 0px 0px 5px rgba(255, 215, 0, 0.3);
}
.output pre, .output code { color: white !important; }

.dark-log-output,
.dark-log-output .jp-OutputArea-output,
.dark-log-output .jp-OutputArea-child,
.dark-log-output .cell-output-ipywidget-background,
.dark-log-output .output_area,
.dark-log-output .jupyter-widgets-output-area,
.dark-log-output .widget-output,
.dark-log-output pre {
    background-color: #111 !important;
    color: white !important;
    border: none !important;
    box-shadow: none !important;
}

.dark-dropdown select {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
}
.dark-dropdown select option {
    background-color: #1e1e1e !important;
    color: white !important;
}
.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}
.dark-toggle:hover { background-color: #4a4a4a !important; }
.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}
.dark-toggle-active:hover { background-color: #388e3c !important; }
.dark-search input {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
    font-family: monospace !important;
}
.dark-search input::placeholder { color: #888 !important; }

.dark-input input {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
    font-family: monospace !important;
    font-size: 14px !important;
}
.dark-input input::placeholder { color: #888 !important; }

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}
body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}
.dark-table th, .dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}
.dark-table th { background-color: #222; color: white; }
.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}
.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}
.header-table th, .header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}
.header-table th { background-color: #222; color: white; }
.header-scroll-box {
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}
</style>
"""))

from daschlab import open_session
from pathlib import Path
from io import BytesIO
from datetime import datetime
from astropy.io import fits
from astropy.time import Time
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import threading
import time
import re
import traceback
import ipywidgets as widgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BASE_DATA_DIR = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data"
)

# Batch mode does a two-phase pass: PHASE_1_PLATE_LIMIT plates for every
# object first, then a second pass finishing each object's remainder.
PHASE_1_PLATE_LIMIT = 750

# STALL detection (replaces the old flat per-phase timeout): if an object's
# download makes literally ZERO progress (no plate completes) for this many
# seconds, the batch gives up waiting on it and moves to the next object.
# This is NOT a cap on how long a phase may take overall -- a slow-but-
# progressing download of 750 real plates can legitimately take much
# longer than this and will NOT be cut off, since every completed plate
# resets the clock. Only genuine stalls (e.g. a hung name-resolution call)
# trigger this.
STALL_TIMEOUT_SECONDS = 300
STALL_CHECK_INTERVAL_SECONDS = 5

def sanitize_folder_name(name):
    """Turns an object name into a safe Windows folder name: strips
    characters that are illegal in Windows paths, then replaces spaces
    with underscores."""
    cleaned = re.sub(r'[<>:"/\\|?*]', '', name)
    cleaned = cleaned.strip()
    cleaned = re.sub(r'\s+', '_', cleaned)
    return cleaned

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def get_dir_size_bytes(path):
    """Sums file sizes under `path` (recursively). Used to report per-object
    storage usage in the batch summary. Returns 0 if the path doesn't exist."""
    total = 0
    if not path.exists():
        return 0
    for f in path.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except OSError:
                pass
    return total

def format_bytes(n):
    """Human-readable byte count, e.g. 1536000 -> '1.46 MB'."""
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def plate_limit_stats(df):
    """Computes plate-limit stats from the manifest's lim_mag columns.
    Called ONCE per object run now (these columns never change mid-run),
    not on every save -- see PERF NOTE inside run_extraction_body()."""
    apass = pd.to_numeric(df.get("lim_mag_apass"), errors="coerce")
    atlas = pd.to_numeric(df.get("lim_mag_atlas"), errors="coerce")
    n_apass = int(apass.notna().sum())
    n_atlas = int(atlas.notna().sum())
    med_apass = f"{apass.median():.2f}" if n_apass else "--"
    med_atlas = f"{atlas.median():.2f}" if n_atlas else "--"
    return n_apass, med_apass, n_atlas, med_atlas

def obs_date_range(df):
    """Computes the earliest/latest observation date from the manifest.
    Also called ONCE per object run now, same reasoning as plate_limit_stats."""
    if "obs_date" not in df.columns:
        return "unknown", "unknown"
    parsed = pd.to_datetime(df["obs_date"], errors="coerce").dropna()
    if parsed.empty:
        return "unknown", "unknown"
    return str(parsed.min().date()), str(parsed.max().date())

def parse_object_list_file(path):
    """Reads a .txt file of object names, one per line. Blank lines and
    lines starting with '#' are skipped. Duplicate names (case-insensitive)
    are dropped, keeping the first occurrence.

    Two line formats are supported:
      Name                     -- resolved via Sesame (name lookup)
      Name|RA_deg|Dec_deg      -- resolved via EXPLICIT coordinates, bypassing
                                   Sesame entirely.

    Returns a list of (name, ra_deg_or_None, dec_deg_or_None) tuples.
    """
    text = Path(path).read_text(encoding="utf-8")
    seen = set()
    entries = []
    for line_num, line in enumerate(text.splitlines(), start=1):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        parts = [p.strip() for p in stripped.split("|")]
        name = parts[0]
        ra_deg = None
        dec_deg = None

        if len(parts) == 3:
            try:
                ra_deg = float(parts[1])
                dec_deg = float(parts[2])
            except ValueError:
                ra_deg = None
                dec_deg = None
        elif len(parts) not in (1, 3):
            name = stripped
            ra_deg = None
            dec_deg = None

        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        entries.append((name, ra_deg, dec_deg))
    return entries


# ============================================================================
# Object name input : type any object daschlab/DASCH can resolve, then click
# Extract Plates. This can be reused as many times as you like for different
# objects, without re-running this cell.
# ============================================================================

target_input = widgets.Text(
    placeholder="e.g. R CrB, V CrA, HD 209458...",
    description="Object:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "60px"},
)
target_input.add_class("dark-input")

extract_button = widgets.Button(
    description="Extract Plates",
    button_style="success",
    layout=widgets.Layout(width="160px", height="32px"),
)

input_status_label = widgets.HTML(value="")

input_container = widgets.HBox(
    [target_input, extract_button, input_status_label],
    layout=widgets.Layout(align_items="center", margin="0 0 8px 0"),
)

# Batch input row -- load a .txt file of object names (optionally with
# explicit coordinates, see parse_object_list_file() above). Batch mode
# runs a two-phase pass: PHASE_1_PLATE_LIMIT plates per object first, then
# a second pass to finish each object's remainder.
batch_path_input = widgets.Text(
    placeholder=r"C:\path\to\objects.txt",
    description="Batch file (.txt):",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "120px"},
)
batch_path_input.add_class("dark-input")

batch_load_button = widgets.Button(
    description="Load & Extract All",
    button_style="info",
    layout=widgets.Layout(width="160px", height="32px"),
)

batch_cancel_button = widgets.Button(
    description="Cancel Batch",
    button_style="danger",
    layout=widgets.Layout(width="120px", height="32px"),
    disabled=True,
)

batch_status_label = widgets.HTML(value="")

batch_container = widgets.HBox(
    [batch_path_input, batch_load_button, batch_cancel_button, batch_status_label],
    layout=widgets.Layout(align_items="center", margin="0 0 12px 0"),
)

# Per-object dashboards are stored here (name -> VBox containing that
# object's pause button / progress bar / summary card / log / visualizer),
# and this dropdown lets you pick which one is currently shown. Selecting
# an object does NOT stop or affect any other object's background thread --
# every panel keeps updating live regardless of whether it's visible.
object_panels = {}

object_selector = widgets.Dropdown(
    options=[],
    description="View object:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "90px"},
)
object_selector.add_class("dark-dropdown")

selector_container = widgets.HBox(
    [object_selector],
    layout=widgets.Layout(margin="0 0 8px 0"),
)

# Batch-level messages (loaded list, per-object start markers, final batch
# report) go here -- always visible, independent of which object's panel
# is currently selected in the dropdown above.
batch_log_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #444", padding="8px",
        max_height="220px", overflow="auto",
    )
)
batch_log_output.add_class("dark-log-output")

# The currently-selected object's panel renders here.
panel_display_area = widgets.Output()

display(input_container, batch_container, selector_container, batch_log_output, panel_display_area)

_active_download_thread = None
_active_batch_thread = None
_batch_cancel_event = threading.Event()
_current_displayed_name = [None]  # Track which panel is currently displayed


def show_panel(name):
    """Renders object_panels[name] into panel_display_area."""
    if name not in object_panels:
        return
    _current_displayed_name[0] = name
    with panel_display_area:
        clear_output(wait=True)
        display(object_panels[name])


def register_panel(name, panel):
    """Adds `panel` to object_panels and to the View Object dropdown's
    options, without necessarily switching the current view. Safe to call
    from the main thread only (this is where widget creation should live)."""
    object_panels[name] = panel
    current_options = list(object_panels.keys())
    if list(object_selector.options) != current_options:
        object_selector.options = current_options


def on_object_selector_change(change):
    name = change["new"]
    if name and name in object_panels:
        show_panel(name)

object_selector.observe(on_object_selector_change, names="value")


def create_panel(target_name):
    """MUST be called from the main/UI thread (i.e. directly inside a
    button-click handler, never from inside a background thread). Builds
    all widgets for one object's dashboard, registers + displays it, and
    returns a dict of the pieces run_extraction_body() will update later."""
    pause_button = widgets.Button(
        description="⏸ Pause",
        button_style="warning",
        layout=widgets.Layout(width="120px", height="36px"),
    )
    progress_html = widgets.HTML(value="<i>Setting up session...</i>")

    top_bar = widgets.HBox(
        [pause_button, progress_html],
        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"),
    )

    summary_html = widgets.HTML(value="")

    log_output = widgets.Output(
        layout=widgets.Layout(
            border="1px solid #444", padding="8px",
            max_height="300px", overflow="auto",
        )
    )
    log_output.add_class("dark-log-output")

    visualizer_slot = widgets.Output()

    panel = widgets.VBox([top_bar, summary_html, log_output, visualizer_slot])

    pause_event = threading.Event()

    def on_pause_clicked(b):
        pause_event.set()
        pause_button.description = "Pausing..."
        pause_button.disabled = True

    pause_button.on_click(on_pause_clicked)

    register_panel(target_name, panel)

    return {
        "pause_button": pause_button,
        "progress_html": progress_html,
        "summary_html": summary_html,
        "log_output": log_output,
        "visualizer_slot": visualizer_slot,
        "pause_event": pause_event,
    }


def run_extraction_body(target_name, panel_widgets, result_container=None, download_limit=None,
                         ra_deg=None, dec_deg=None):
    """Runs the full download-then-visualize pipeline for one object,
    updating the already-built widgets in panel_widgets (see create_panel()
    above). Safe to call from a background thread. Returns the background
    download thread, or None if the target could not be resolved at all.

    ra_deg / dec_deg: optional. If BOTH are provided, the target is resolved
    via explicit coordinates, bypassing Sesame name resolution entirely.

    result_container: optional dict. If provided, its "heartbeat" key gets
    updated with the current time every time a plate finishes downloading
    (see run_downloads() below) -- this is what the batch runner polls to
    distinguish "slow but working" from "genuinely stalled"."""
    global _active_download_thread

    pause_button = panel_widgets["pause_button"]
    progress_html = panel_widgets["progress_html"]
    summary_html = panel_widgets["summary_html"]
    log_output = panel_widgets["log_output"]
    visualizer_slot = panel_widgets["visualizer_slot"]
    pause_event = panel_widgets["pause_event"]

    target_slug = sanitize_folder_name(target_name)
    session_dir = BASE_DATA_DIR / target_slug
    session_dir.mkdir(parents=True, exist_ok=True)

    cutout_dir = session_dir / "cutouts"
    manifest_path = session_dir / "plate_manifest.csv"
    summary_path = session_dir / "plate_manifest_summary.csv"
    text_summary_path = session_dir / f"{target_slug}_Cutout_Download_Summary.txt"

    def render_progress(n_done, total, start_time, status_word="Downloading", color="#4CAF50"):
        elapsed = time.monotonic() - start_time
        pct = (n_done / total * 100) if total else 0
        rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
        remaining = ((total - n_done) / rate) if rate > 0 else None

        bar_width = 380
        filled = int(bar_width * pct / 100)

        progress_html.value = f"""
        <div style="font-family: monospace; font-size: 14px; color: white; line-height: 1.6;">
          <div style="display:flex; align-items:center;">
            <div style="width:{bar_width}px; height:18px; background:#333;
                        border-radius:4px; overflow:hidden; margin-right:12px;">
              <div style="width:{filled}px; height:100%; background:{color};
                          transition: width 0.3s;"></div>
            </div>
            <b>{pct:5.1f}%</b>
          </div>
          <div style="margin-top:4px;">
            {status_word} &nbsp;
            <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
            <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
            Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp;
            ETA <b>{format_eta(remaining)}</b>
          </div>
        </div>
        """

    def render_summary_card(title, title_color, stat_items, footnote=None):
        cards = ""
        for label, value, accent in stat_items:
            display_value = f"{value:,}" if isinstance(value, int) else value
            cards += f"""
            <div style="background:#1a1a1a; border:1px solid {accent};
                        border-radius:8px; padding:10px 18px; margin:4px 8px 4px 0;
                        min-width:110px; text-align:center; flex:1;">
              <div style="font-size:22px; font-weight:bold; color:{accent};
                          font-family: monospace;">{display_value}</div>
              <div style="font-size:11px; color:#aaa; margin-top:3px;
                          text-transform:uppercase; letter-spacing:0.5px;">{label}</div>
            </div>
            """
        footnote_html = (
            f'<div style="color:#888; font-size:12px; margin-top:8px; font-family: monospace;">{footnote}</div>'
            if footnote else ""
        )
        summary_html.value = f"""
        <div style="background:#111; border:1px solid #333; border-radius:10px;
                    padding:12px 14px; margin-bottom:12px;">
          <div style="font-size:15px; font-weight:bold; color:{title_color};
                      font-family: monospace; margin-bottom:8px;">{title}</div>
          <div style="display:flex; flex-wrap:wrap;">{cards}</div>
          {footnote_html}
        </div>
        """

    def compute_summary_fields(manifest_df, MAX_WORKERS, plate_limit_cache, date_range_cache, disk_count,
                                extraction_start_ts=None, extraction_end_ts=None,
                                extraction_seconds=None, run_status="not started",
                                this_run_counts=None):
        counts = manifest_df["status"].value_counts().to_dict()
        n_apass, med_apass, n_atlas, med_atlas = plate_limit_cache
        earliest_date, latest_date = date_range_cache

        total = len(manifest_df)
        def pct(n):
            return f"{(n / total * 100):.1f}%" if total else "0.0%"

        this_run_counts = this_run_counts or {"downloaded": 0, "unavailable": 0, "error": 0}

        if extraction_seconds is not None:
            m, s = divmod(int(extraction_seconds), 60)
            h, m = divmod(m, 60)
            duration_human = f"{h:d}h {m:02d}m {s:02d}s" if h else f"{m:d}m {s:02d}s"
            rate_per_sec = (
                (this_run_counts["downloaded"] + this_run_counts["unavailable"] + this_run_counts["error"])
                / extraction_seconds
            ) if extraction_seconds > 0 else 0
        else:
            duration_human = "n/a"
            rate_per_sec = 0

        return {
            "target_name": target_name,
            "target_folder_slug": target_slug,
            "session_directory": str(session_dir),
            "cutout_directory": str(cutout_dir),
            "manifest_file": str(manifest_path),
            "summary_file": str(summary_path),
            "text_summary_file": str(text_summary_path),
            "run_status": run_status,
            "extraction_start_time": extraction_start_ts.isoformat(timespec="seconds") if extraction_start_ts else "n/a",
            "extraction_end_time": extraction_end_ts.isoformat(timespec="seconds") if extraction_end_ts else "n/a",
            "extraction_duration_seconds": round(extraction_seconds, 2) if extraction_seconds is not None else "n/a",
            "extraction_duration_human": duration_human,
            "extraction_avg_rate_plates_per_sec": round(rate_per_sec, 3),
            "max_concurrent_workers_used": MAX_WORKERS,
            "this_run_downloaded": this_run_counts.get("downloaded", 0),
            "this_run_unavailable": this_run_counts.get("unavailable", 0),
            "this_run_error": this_run_counts.get("error", 0),
            "total_exposures_all_time": total,
            "fits_files_on_disk": disk_count,
            "downloaded_all_time": counts.get("downloaded", 0),
            "downloaded_pct": pct(counts.get("downloaded", 0)),
            "unavailable_all_time": counts.get("unavailable", 0),
            "unavailable_pct": pct(counts.get("unavailable", 0)),
            "error_all_time": counts.get("error", 0),
            "error_pct": pct(counts.get("error", 0)),
            "not_attempted_all_time": counts.get("not_attempted", 0),
            "not_attempted_pct": pct(counts.get("not_attempted", 0)),
            "plates_with_lim_mag_apass": n_apass,
            "median_lim_mag_apass": med_apass,
            "plates_with_lim_mag_atlas": n_atlas,
            "median_lim_mag_atlas": med_atlas,
            "earliest_observation_date": earliest_date,
            "latest_observation_date": latest_date,
            "summary_generated_at": datetime.now().isoformat(timespec="seconds"),
        }

    def build_summary_df(fields):
        return pd.DataFrame(list(fields.items()), columns=["metric", "value"])

    def build_text_summary(fields):
        lines = []
        lines.append("=" * 70)
        lines.append(f"DASCH PLATE CUTOUT DOWNLOAD SUMMARY")
        lines.append(f"Object : {fields['target_name']}")
        lines.append("=" * 70)
        lines.append("")
        lines.append(f"Run status                     : {fields['run_status']}")
        lines.append(f"Summary generated at           : {fields['summary_generated_at']}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("TIMING")
        lines.append("-" * 70)
        lines.append(f"Extraction start time           : {fields['extraction_start_time']}")
        lines.append(f"Extraction end time              : {fields['extraction_end_time']}")
        lines.append(f"Extraction duration (seconds)   : {fields['extraction_duration_seconds']}")
        lines.append(f"Extraction duration (human)      : {fields['extraction_duration_human']}")
        lines.append(f"Average rate (plates/sec)        : {fields['extraction_avg_rate_plates_per_sec']}")
        lines.append(f"Max concurrent download workers  : {fields['max_concurrent_workers_used']}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("THIS RUN")
        lines.append("-" * 70)
        lines.append(f"Downloaded this run              : {fields['this_run_downloaded']:,}")
        lines.append(f"Unavailable this run              : {fields['this_run_unavailable']:,}")
        lines.append(f"Errored this run                  : {fields['this_run_error']:,}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("ALL-TIME TOTALS (this object, across every run)")
        lines.append("-" * 70)
        lines.append(f"Total exposures known             : {fields['total_exposures_all_time']:,}")
        lines.append(f"FITS files currently on disk       : {fields['fits_files_on_disk']:,}")
        lines.append(f"Downloaded                         : {fields['downloaded_all_time']:,}  ({fields['downloaded_pct']})")
        lines.append(f"Unavailable (unscanned)            : {fields['unavailable_all_time']:,}  ({fields['unavailable_pct']})")
        lines.append(f"Errored                            : {fields['error_all_time']:,}  ({fields['error_pct']})")
        lines.append(f"Not attempted yet                  : {fields['not_attempted_all_time']:,}  ({fields['not_attempted_pct']})")
        lines.append("")
        lines.append("-" * 70)
        lines.append("PLATE LIMITS (limiting magnitude)")
        lines.append("-" * 70)
        lines.append(f"Plates with lim_mag_apass          : {fields['plates_with_lim_mag_apass']:,}")
        lines.append(f"Median lim_mag_apass                : {fields['median_lim_mag_apass']}")
        lines.append(f"Plates with lim_mag_atlas          : {fields['plates_with_lim_mag_atlas']:,}")
        lines.append(f"Median lim_mag_atlas                : {fields['median_lim_mag_atlas']}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("OBSERVATION DATE COVERAGE")
        lines.append("-" * 70)
        lines.append(f"Earliest observation date          : {fields['earliest_observation_date']}")
        lines.append(f"Latest observation date             : {fields['latest_observation_date']}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("FILE LOCATIONS")
        lines.append("-" * 70)
        lines.append(f"Session directory                   : {fields['session_directory']}")
        lines.append(f"Cutout directory (the .fits files)  : {fields['cutout_directory']}")
        lines.append(f"Manifest CSV (one row per exposure) : {fields['manifest_file']}")
        lines.append(f"Summary CSV (this data, machine-readable) : {fields['summary_file']}")
        lines.append(f"This text summary                   : {fields['text_summary_file']}")
        lines.append("")
        lines.append("=" * 70)
        return "\n".join(lines)

    try:
        with log_output:
            sess = open_session(str(session_dir))

            if ra_deg is not None and dec_deg is not None:
                print(f"Resolving via explicit coordinates (ra={ra_deg:.5f}, dec={dec_deg:.5f}) "
                      f"-- skipping name lookup.")
                sess.select_target(ra_deg=ra_deg, dec_deg=dec_deg)
            else:
                sess.select_target(target_name)

            sess.select_refcat("apass")

            exposures = sess.exposures()
            print(f"Target : {target_name}")
            print(f"Total exposures : {len(exposures)}")

            try:
                exposures_df = exposures.to_pandas()
            except Exception as e:
                print(f"Could not convert exposures table to pandas ({e}); manifest will have fewer columns.")
                exposures_df = pd.DataFrame(index=range(len(exposures)))

            exposures_df.insert(0, "exposure_index", range(len(exposures_df)))

            if manifest_path.exists():
                manifest_df = pd.read_csv(manifest_path)
                print(f"Loaded existing manifest with {len(manifest_df)} rows.")
            else:
                manifest_df = exposures_df.copy()
                manifest_df["status"] = "not_attempted"
                manifest_df["filename"] = ""
                manifest_df["error_message"] = ""
                manifest_df["last_updated"] = ""

            front_cols = [c for c in [
                "exposure_index", "status", "filename",
                "lim_mag_apass", "lim_mag_atlas",
                "error_message", "last_updated",
            ] if c in manifest_df.columns]
            other_cols = [c for c in manifest_df.columns if c not in front_cols]
            manifest_df = manifest_df[front_cols + other_cols]

            manifest_df = manifest_df.set_index("exposure_index", drop=False)

            existing_files = {f.name for f in cutout_dir.glob("*.fits")} if cutout_dir.exists() else set()
            if existing_files and "filename" in manifest_df.columns:
                on_disk_mask = manifest_df["filename"].apply(
                    lambda f: Path(f).name in existing_files if isinstance(f, str) and f else False
                )
                reconciled = (manifest_df["status"] != "downloaded") & on_disk_mask
                if reconciled.any():
                    manifest_df.loc[reconciled, "status"] = "downloaded"
                    print(f"Reconciled {reconciled.sum()} rows already on disk but not marked downloaded.")

            remaining_indices = manifest_df.index[manifest_df["status"] != "downloaded"].tolist()
            print(f"{len(remaining_indices)} cutouts remaining to download "
                  f"(out of {len(manifest_df)} total).")

            if download_limit is not None:
                remaining_indices = remaining_indices[:download_limit]
                print(f"This run is limited to {len(remaining_indices)} plates "
                      f"(download_limit={download_limit}).")

    except Exception as e:
        with log_output:
            print(f"\n Could not resolve or query target '{target_name}': {e}")
            traceback.print_exc()
        input_status_label.value = f"<span style='color:#EF5350;'>Failed to resolve '{target_name}'. See log above.</span>"
        extract_button.disabled = False
        return None

    input_status_label.value = ""

    # PERF: computed ONCE here, reused for every save this run instead of
    # being recalculated from the full manifest on every single save.
    plate_limit_cache = plate_limit_stats(manifest_df)
    date_range_cache = obs_date_range(manifest_df)
    initial_disk_count = len(existing_files)

    _n_apass, _med_apass, _n_atlas, _med_atlas = plate_limit_cache
    render_summary_card(
        title=f"{target_name} - Manifest Overview (before this run)",
        title_color="#90caf9",
        stat_items=[
            ("Total", len(manifest_df), "#90caf9"),
            ("Downloaded", int((manifest_df["status"] == "downloaded").sum()), "#4CAF50"),
            ("Unavailable", int((manifest_df["status"] == "unavailable").sum()), "#FFA726"),
            ("Errored", int((manifest_df["status"] == "error").sum()), "#EF5350"),
            ("Not attempted", int((manifest_df["status"] == "not_attempted").sum()), "#9E9E9E"),
            ("Plate limit (APASS)", f"{_n_apass:,} plates | med {_med_apass}", "#BA68C8"),
            ("Plate limit (ATLAS)", f"{_n_atlas:,} plates | med {_med_atlas}", "#4DD0E1"),
        ],
        footnote=f"{len(remaining_indices):,} plates queued for this run.",
    )

    total_remaining = max(len(remaining_indices), 1)
    render_progress(0, total_remaining, time.monotonic(), status_word="Starting...")

    MAX_WORKERS = 6

    def fetch_one(i):
        try:
            result = sess.cutout(i)
            if result is None:
                return i, "unavailable", None, "no cutout available (likely unscanned)"
            return i, "downloaded", str(result), None
        except Exception as e:
            return i, "error", None, str(e)

    def save_manifest():
        try:
            manifest_df.reset_index(drop=True).sort_values("exposure_index").to_csv(
                manifest_path, index=False
            )
            return True
        except Exception as e:
            with log_output:
                print(f"Couldn't save manifest right now ({e}). "
                      f"If it's open in Excel, close it - will retry automatically.")
            return False

    def save_summary(extraction_start_ts=None, extraction_end_ts=None,
                      extraction_seconds=None, run_status="not started",
                      this_run_counts=None, disk_count=None):
        try:
            if disk_count is None:
                disk_count = len(list(cutout_dir.glob("*.fits"))) if cutout_dir.exists() else 0
            fields = compute_summary_fields(
                manifest_df, MAX_WORKERS, plate_limit_cache, date_range_cache, disk_count,
                extraction_start_ts=extraction_start_ts,
                extraction_end_ts=extraction_end_ts,
                extraction_seconds=extraction_seconds,
                run_status=run_status,
                this_run_counts=this_run_counts,
            )
            build_summary_df(fields).to_csv(summary_path, index=False)
            text_summary_path.write_text(build_text_summary(fields), encoding="utf-8")
            return True
        except Exception as e:
            with log_output:
                print(f"Couldn't save summary right now ({e}). Will retry automatically.")
            return False

    def save_all(extraction_start_ts=None, extraction_end_ts=None,
                 extraction_seconds=None, run_status="not started",
                 this_run_counts=None, disk_count=None):
        ok1 = save_manifest()
        ok2 = save_summary(extraction_start_ts, extraction_end_ts,
                            extraction_seconds, run_status, this_run_counts, disk_count)
        return ok1 and ok2

    def build_and_show_visualizer():
        cutouts = sorted(cutout_dir.glob("*.fits"))

        date_lookup = {}
        if manifest_path.exists():
            manifest_df_for_dates = pd.read_csv(manifest_path)
            if "filename" in manifest_df_for_dates.columns and "obs_date" in manifest_df_for_dates.columns:
                for _, row in manifest_df_for_dates.iterrows():
                    fname = row.get("filename")
                    odate = row.get("obs_date")
                    if isinstance(fname, str) and fname and pd.notna(odate):
                        date_lookup[Path(fname).name] = str(odate)

        def get_plate_date_from_header(path):
            try:
                hdr = fits.getheader(path)
            except Exception:
                return path.stem
            for key in ("DATE-OBS", "DATE_OBS", "DATEORIG"):
                if key in hdr and hdr[key]:
                    return str(hdr[key])
            for key in ("MJD",):
                if key in hdr and hdr[key]:
                    try:
                        return Time(float(hdr[key]), format="mjd").iso.split(" ")[0]
                    except Exception:
                        pass
            for key in ("JD",):
                if key in hdr and hdr[key]:
                    try:
                        return Time(float(hdr[key]), format="jd").iso.split(" ")[0]
                    except Exception:
                        pass
            return path.stem

        def get_plate_date(path):
            cached = date_lookup.get(path.name)
            if cached is not None:
                return cached
            return get_plate_date_from_header(path)

        dropdown_options = [
            (f"{get_plate_date(f)} - {f.name}", f) for f in cutouts
        ]
        all_dropdown_options = dropdown_options

        search_box = widgets.Text(
            placeholder="Search by filename or date...",
            layout=widgets.Layout(width="1200px", margin="0 0 6px 0")
        )
        search_box.add_class("dark-search")
        search_container = widgets.HBox(
            [search_box], layout=widgets.Layout(justify_content="center", width="100%")
        )

        dropdown = widgets.Select(
            options=dropdown_options, rows=12, description="",
            layout=widgets.Layout(width="1200px", height="300px")
        )
        dropdown.add_class("dark-dropdown")
        dropdown_container = widgets.HBox(
            [dropdown], layout=widgets.Layout(justify_content="center", width="100%")
        )

        toggle_header = widgets.ToggleButton(value=False, description="Show Header",
                                              layout=widgets.Layout(width="160px"))
        toggle_data = widgets.ToggleButton(value=False, description="Show Pixel Data [SLOW]",
                                            layout=widgets.Layout(width="180px"))
        toggle_header.add_class("dark-toggle")
        toggle_data.add_class("dark-toggle")

        def make_label_handler(toggle, label):
            def handler(change):
                if change["new"]:
                    toggle.description = f"Hide {label}"
                    toggle.add_class("dark-toggle-active")
                else:
                    toggle.description = f"Show {label}"
                    toggle.remove_class("dark-toggle-active")
            return handler

        toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
        toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")

        copy_button = widgets.Button(description="Copy Filename", layout=widgets.Layout(width="180px"))
        copy_button.add_class("dark-toggle")
        copy_status_label = widgets.HTML(value="")

        def copy_to_clipboard(text):
            escaped = text.replace("\\", "\\\\").replace('"', '\\"')
            display(Javascript(f'''
                navigator.clipboard.writeText("{escaped}").catch(function(err) {{
                    console.error("Clipboard copy failed:", err);
                }});
            '''))

        def on_copy_clicked(b):
            if current_path[0] is None:
                copy_status_label.value = "<span style='color:#EF5350;'>No plate selected yet.</span>"
                return
            filename = current_path[0].name
            copy_to_clipboard(filename)
            copy_status_label.value = f"<span style='color:#66bb6a;'>✅ Copied: {filename}</span>"

        copy_button.on_click(on_copy_clicked)

        toggles_container = widgets.HBox(
            [toggle_header, toggle_data, copy_button, copy_status_label],
            layout=widgets.Layout(justify_content="center", width="100%", margin="10px 0px")
        )

        viz_image_output = widgets.Output(layout=widgets.Layout(
            padding="10px", width="100%", display="flex",
            justify_content="center", align_items="center"
        ))
        viz_image_output.add_class("dark-output")
        image_container = widgets.HBox([viz_image_output], layout=widgets.Layout(justify_content="center", width="100%"))

        headers_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
        headers_output.add_class("dark-output")
        headers_container = widgets.HBox([headers_output], layout=widgets.Layout(justify_content="center", width="100%"))

        data_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
        data_output.add_class("dark-output")
        data_container = widgets.HBox([data_output], layout=widgets.Layout(justify_content="center", width="100%"))

        current_path = [None]
        current_data = [None]

        def on_search_change(change):
            query = change["new"].strip().lower()
            if not query:
                dropdown.options = all_dropdown_options
                return
            filtered = [(label, path) for label, path in all_dropdown_options if query in label.lower()]
            dropdown.options = filtered if filtered else [("No matches", None)]

        search_box.observe(on_search_change, names="value")

        def render_header_panel():
            with headers_output:
                clear_output(wait=True)
                if not toggle_header.value or current_path[0] is None:
                    return
                header = fits.getheader(current_path[0])
                header_df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])
                header_html = header_df.to_html(classes="header-table", border=0, index=False)
                display(HTML("<h3 style='color:white;text-align:center;'>FITS Header</h3>"))
                display(HTML(f'<div class="header-scroll-box">{header_html}</div>'))

        def render_data_panel():
            with data_output:
                clear_output(wait=True)
                if not toggle_data.value or current_data[0] is None:
                    return
                df = pd.DataFrame(current_data[0])
                table_html = df.to_html(classes="dark-table", border=0)
                display(HTML("<h3 style='color:white;text-align:center;'>Pixel Data</h3>"))
                display(HTML(f'<div class="table-scroll-box">{table_html}</div>'))

        def show_plate(change):
            if change["new"] is None:
                return
            current_path[0] = change["new"]
            current_data[0] = fits.getdata(current_path[0])
            copy_status_label.value = ""

            with viz_image_output:
                clear_output(wait=True)
                fig = plt.figure(figsize=(12, 10), facecolor="#111111")
                gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 1.0, 0.05], wspace=0.03)
                ax_spacer = fig.add_subplot(gs[0])
                ax = fig.add_subplot(gs[1])
                cax = fig.add_subplot(gs[2])
                ax_spacer.axis("off")
                fig.patch.set_facecolor("#111111")
                ax.set_facecolor("#111111")

                im = ax.imshow(current_data[0], origin="lower", cmap="gray_r")

                cbar = fig.colorbar(im, cax=cax)
                cbar.set_label("Plate Intensity", color="white", fontsize=14)
                cbar.ax.tick_params(colors="white")

                ax.set_title(current_path[0].stem, fontsize=28, color="white", pad=16)
                ax.set_xlabel("X Pixel", color="white")
                ax.set_ylabel("Y Pixel", color="white")
                ax.tick_params(colors="white")

                buf = BytesIO()
                fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
                plt.close(fig)
                buf.seek(0)
                display(widgets.Image(value=buf.read(), format="png"))

            render_header_panel()
            render_data_panel()

        dropdown.observe(show_plate, names="value")
        toggle_header.observe(lambda change: render_header_panel(), names="value")
        toggle_data.observe(lambda change: render_data_panel(), names="value")

        with visualizer_slot:
            clear_output(wait=True)
            print(f"Found {len(cutouts)} cutouts")
            display(
                search_container,
                dropdown_container,
                toggles_container,
                image_container,
                headers_container,
                data_container,
            )
            if cutouts:
                show_plate({"new": cutouts[0]})

    def run_downloads():
        counts = {"downloaded": 0, "unavailable": 0, "error": 0}
        n_done = 0
        extraction_start_ts = datetime.now()
        start_time = time.monotonic()
        stopped_early = False

        ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
        futures = {ex.submit(fetch_one, i): i for i in remaining_indices}
        pending = set(futures.keys())

        try:
            while pending:
                if pause_event.is_set():
                    stopped_early = True
                    with log_output:
                        print("\n⏸ Pause requested - no new downloads will start.")
                        print("   A few in-flight downloads may finish in the background; that's fine.")
                    ex.shutdown(wait=False, cancel_futures=True)
                    break

                done, pending = wait(pending, timeout=0.2, return_when=FIRST_COMPLETED)
                for fut in done:
                    i, status, filename, err_msg = fut.result()
                    manifest_df.loc[i, "status"] = status
                    manifest_df.loc[i, "filename"] = filename or ""
                    manifest_df.loc[i, "error_message"] = err_msg or ""
                    manifest_df.loc[i, "last_updated"] = datetime.now().isoformat(timespec="seconds")
                    counts[status] += 1
                    n_done += 1

                if done:
                    render_progress(n_done, len(futures), start_time)
                    # STALL-DETECTION: mark that real progress just happened,
                    # so the batch runner's stall check (see run_one_phase)
                    # knows this object is still actively working, however
                    # slow, and should NOT be given up on.
                    if result_container is not None:
                        result_container.setdefault("heartbeat", {})["last_update"] = time.monotonic()
                    estimated_disk_count = initial_disk_count + counts["downloaded"]
                    save_all(extraction_start_ts, datetime.now(),
                              time.monotonic() - start_time, "in progress", counts,
                              disk_count=estimated_disk_count)
            else:
                ex.shutdown(wait=True)

        except Exception as e:
            stopped_early = True
            with log_output:
                print(f"\n Download thread hit an unexpected error: {e}")
                traceback.print_exc()

        finally:
            extraction_end_ts = datetime.now()
            extraction_seconds = time.monotonic() - start_time
            run_status = "paused" if stopped_early else "completed"

            cutouts_now = sorted(cutout_dir.glob("*.fits")) if cutout_dir.exists() else []
            final_disk_count = len(cutouts_now)

            saved = save_all(extraction_start_ts, extraction_end_ts,
                              extraction_seconds, run_status, counts,
                              disk_count=final_disk_count)
            if not saved:
                time.sleep(1.0)
                save_all(extraction_start_ts, extraction_end_ts,
                          extraction_seconds, run_status, counts,
                          disk_count=final_disk_count)

            final_word = "Paused" if stopped_early else "Done"
            render_progress(n_done, len(futures), start_time,
                             status_word=final_word,
                             color="#FFA726" if stopped_early else "#4CAF50")
            pause_button.description = final_word
            pause_button.disabled = True

            _counts_after = manifest_df["status"].value_counts().to_dict()
            _n_apass2, _med_apass2, _n_atlas2, _med_atlas2 = plate_limit_cache
            title_word = "⏸ Paused" if stopped_early else "Final Summary"
            title_color = "#FFA726" if stopped_early else "#4CAF50"
            render_summary_card(
                title=f"{target_name} - {title_word} - this run: "
                      f"{counts['downloaded']:,} downloaded, "
                      f"{counts['unavailable']:,} unavailable, "
                      f"{counts['error']:,} errored, "
                      f"took {format_eta(extraction_seconds)}",
                title_color=title_color,
                stat_items=[
                    ("Total exposures", len(manifest_df), "#90caf9"),
                    ("On disk", final_disk_count, "#64B5F6"),
                    ("Downloaded", _counts_after.get("downloaded", 0), "#4CAF50"),
                    ("Unavailable", _counts_after.get("unavailable", 0), "#FFA726"),
                    ("Errored", _counts_after.get("error", 0), "#EF5350"),
                    ("Not attempted", _counts_after.get("not_attempted", 0), "#9E9E9E"),
                    ("Plate limit (APASS)", f"{_n_apass2:,} plates | med {_med_apass2}", "#BA68C8"),
                    ("Plate limit (ATLAS)", f"{_n_atlas2:,} plates | med {_med_atlas2}", "#4DD0E1"),
                ],
                footnote=(
                    ("Safe to close now. Rerun this cell anytime to pick back up. &nbsp;|&nbsp; "
                     if stopped_early else "")
                    + f"Manifest: {manifest_path}  |  Summary: {summary_path}  |  Text report: {text_summary_path}"
                ),
            )

            with log_output:
                print(f"\n{'Paused' if stopped_early else 'Finished'} - took {format_eta(extraction_seconds)}")
                print(f"See summary card above, plus:")
                print(f"  {summary_path.name} (CSV)")
                print(f"  {text_summary_path.name} (text report)")

            build_and_show_visualizer()
            extract_button.disabled = False

            if result_container is not None:
                result_container["result"] = {
                    "target_name": target_name,
                    "status": run_status,
                    "duration_seconds": extraction_seconds,
                    "downloaded": counts["downloaded"],
                    "unavailable": counts["unavailable"],
                    "error": counts["error"],
                }

    global _active_download_thread
    _active_download_thread = threading.Thread(target=run_downloads, daemon=True)
    _active_download_thread.start()
    return _active_download_thread


def on_extract_clicked(b):
    global _active_download_thread

    target_name = target_input.value.strip()
    if not target_name:
        input_status_label.value = "<span style='color:#EF5350;'>Please enter an object name.</span>"
        return
    if _active_download_thread is not None and _active_download_thread.is_alive():
        input_status_label.value = "<span style='color:#FFA726;'>⚠️ An extraction is already running - wait for it to finish.</span>"
        return
    if _active_batch_thread is not None and _active_batch_thread.is_alive():
        input_status_label.value = "<span style='color:#FFA726;'>⚠️ A batch is currently running - wait for it to finish, or cancel it.</span>"
        return

    extract_button.disabled = True
    input_status_label.value = f"<span style='color:#4DD0E1;'>Extracting {target_name}...</span>"

    panel_widgets = create_panel(target_name)
    show_panel(target_name)
    object_selector.value = target_name

    threading.Thread(target=run_extraction_body, args=(target_name, panel_widgets), daemon=True).start()


extract_button.on_click(on_extract_clicked)


# ============================================================================
# Batch extraction from a .txt file of object names -- two-phase pattern:
# Phase 1 gets up to PHASE_1_PLATE_LIMIT plates for every object, then
# Phase 2 loops back through the same list and finishes each object's
# remaining plates. The batch report tracks Phase 1 and Phase 2 SEPARATELY
# per object AND rolls each phase up into its own all-objects total, plus
# a final grand-total combining both phases across every object.
# ============================================================================

def build_verbose_batch_report(entries, phase_results, batch_start, batch_end, batch_duration):
    """Builds the full verbose text report. phase_results is
    {name: {1: result_dict_or_None, 2: result_dict_or_None}}, where each
    result_dict has keys: status, duration_seconds, downloaded, unavailable,
    error."""
    lines = []
    lines.append("=" * 78)
    lines.append("BATCH PLATE CUTOUT EXTRACTION SUMMARY (two-phase, verbose)")
    lines.append("=" * 78)
    lines.append("")
    lines.append(f"Batch start time      : {batch_start.isoformat(timespec='seconds')}")
    lines.append(f"Batch end time        : {batch_end.isoformat(timespec='seconds')}")
    lines.append(f"Batch total duration  : {format_eta(batch_duration)} ({batch_duration:.1f}s)")
    lines.append(f"Objects requested     : {len(entries)}")
    lines.append(f"Phase 1 plate limit   : {PHASE_1_PLATE_LIMIT}")
    lines.append(f"Stall detection threshold : {STALL_TIMEOUT_SECONDS}s of zero progress "
                 f"(NOT a cap on total phase duration -- slow-but-working downloads are never cut off)")
    lines.append("")

    # Pre-compute per-object storage and combined totals for the quick table.
    per_object_storage = {}
    combined_by_name = {}
    for name, ra_deg, dec_deg in entries:
        cutout_dir = BASE_DATA_DIR / sanitize_folder_name(name) / "cutouts"
        per_object_storage[name] = get_dir_size_bytes(cutout_dir)

        p1 = phase_results.get(name, {}).get(1)
        p2 = phase_results.get(name, {}).get(2)
        dl = (p1["downloaded"] if p1 else 0) + (p2["downloaded"] if p2 else 0)
        un = (p1["unavailable"] if p1 else 0) + (p2["unavailable"] if p2 else 0)
        er = (p1["error"] if p1 else 0) + (p2["error"] if p2 else 0)
        secs = (p1["duration_seconds"] if p1 else 0) + (p2["duration_seconds"] if p2 else 0)
        combined_by_name[name] = {"downloaded": dl, "unavailable": un, "error": er, "duration_seconds": secs}

    lines.append("-" * 78)
    lines.append("QUICK SUMMARY (combined across both phases, per object)")
    lines.append("-" * 78)
    lines.append(f"{'Object':<28}{'Downloaded':>11}{'Unavail':>9}{'Errored':>9}{'Time':>10}{'Storage':>12}")

    grand_downloaded = grand_unavailable = grand_error = 0
    grand_seconds = 0.0
    grand_storage_bytes = 0

    for name, ra_deg, dec_deg in entries:
        c = combined_by_name[name]
        size_bytes = per_object_storage[name]
        grand_downloaded += c["downloaded"]
        grand_unavailable += c["unavailable"]
        grand_error += c["error"]
        grand_seconds += c["duration_seconds"]
        grand_storage_bytes += size_bytes
        lines.append(
            f"{name:<28}{c['downloaded']:>11,}{c['unavailable']:>9,}{c['error']:>9,}"
            f"{format_eta(c['duration_seconds']):>10}{format_bytes(size_bytes):>12}"
        )

    lines.append("-" * 78)
    lines.append(
        f"{'TOTAL':<28}{grand_downloaded:>11,}{grand_unavailable:>9,}{grand_error:>9,}"
        f"{format_eta(grand_seconds):>10}{format_bytes(grand_storage_bytes):>12}"
    )
    lines.append("")

    # Phase-level rollups across ALL objects: each phase gets its own
    # aggregate block, computed by summing every object's result for that
    # specific phase.
    def phase_rollup(phase_number):
        dl = un = er = 0
        secs = 0.0
        n_objects_with_data = 0
        for name, _ra, _dec in entries:
            r = phase_results.get(name, {}).get(phase_number)
            if r is None:
                continue
            n_objects_with_data += 1
            dl += r["downloaded"]
            un += r["unavailable"]
            er += r["error"]
            secs += r["duration_seconds"]
        attempted = dl + un + er
        avg = (secs / attempted) if attempted > 0 else None
        return {
            "objects": n_objects_with_data,
            "downloaded": dl, "unavailable": un, "error": er,
            "attempted": attempted, "duration_seconds": secs, "avg_time_per_plate": avg,
        }

    phase1_roll = phase_rollup(1)
    phase2_roll = phase_rollup(2)

    phase1_avg_str = f"{phase1_roll['avg_time_per_plate']:.3f}s" if phase1_roll['avg_time_per_plate'] is not None else "n/a"
    phase2_avg_str = f"{phase2_roll['avg_time_per_plate']:.3f}s" if phase2_roll['avg_time_per_plate'] is not None else "n/a"

    lines.append("=" * 78)
    lines.append(f"PHASE 1 TOTALS (all objects, up to {PHASE_1_PLATE_LIMIT} plates each)")
    lines.append("=" * 78)
    lines.append(f"Objects run in this phase   : {phase1_roll['objects']:,} / {len(entries):,}")
    lines.append(f"Total downloaded            : {phase1_roll['downloaded']:,}")
    lines.append(f"Total unavailable           : {phase1_roll['unavailable']:,}")
    lines.append(f"Total errored               : {phase1_roll['error']:,}")
    lines.append(f"Total plates attempted      : {phase1_roll['attempted']:,}")
    lines.append(f"Total time (Phase 1 only)   : {format_eta(phase1_roll['duration_seconds'])} "
                 f"({phase1_roll['duration_seconds']:.1f}s)")
    lines.append(f"Avg time per plate          : {phase1_avg_str}")
    lines.append("")

    lines.append("=" * 78)
    lines.append("PHASE 2 TOTALS (all objects, remaining plates)")
    lines.append("=" * 78)
    lines.append(f"Objects run in this phase   : {phase2_roll['objects']:,} / {len(entries):,}")
    lines.append(f"Total downloaded            : {phase2_roll['downloaded']:,}")
    lines.append(f"Total unavailable           : {phase2_roll['unavailable']:,}")
    lines.append(f"Total errored               : {phase2_roll['error']:,}")
    lines.append(f"Total plates attempted      : {phase2_roll['attempted']:,}")
    lines.append(f"Total time (Phase 2 only)   : {format_eta(phase2_roll['duration_seconds'])} "
                 f"({phase2_roll['duration_seconds']:.1f}s)")
    lines.append(f"Avg time per plate          : {phase2_avg_str}")
    lines.append("")

    # Verbose per-object sections (unchanged structure from before).
    lines.append("=" * 78)
    lines.append("PER-OBJECT DETAIL (Phase 1 vs Phase 2, separately)")
    lines.append("=" * 78)

    for name, ra_deg, dec_deg in entries:
        coord_note = f"  (resolved via coordinates: {ra_deg:.5f}, {dec_deg:.5f})" if ra_deg is not None else ""
        lines.append("")
        lines.append("-" * 78)
        lines.append(f"OBJECT: {name}{coord_note}")
        lines.append("-" * 78)

        for phase_num, phase_label in [
            (1, f"Phase 1 (up to {PHASE_1_PLATE_LIMIT} plates)"),
            (2, "Phase 2 (remaining plates)"),
        ]:
            r = phase_results.get(name, {}).get(phase_num)
            lines.append(f"  {phase_label}:")
            if r is None:
                lines.append(f"    Status                : not run")
                continue
            attempted = r["downloaded"] + r["unavailable"] + r["error"]
            avg_time = (r["duration_seconds"] / attempted) if attempted > 0 else None
            avg_time_str = f"{avg_time:.3f}s" if avg_time is not None else "n/a"
            lines.append(f"    Status                : {r['status']}")
            lines.append(f"    Duration              : {format_eta(r['duration_seconds'])} ({r['duration_seconds']:.2f}s)")
            lines.append(f"    Downloaded            : {r['downloaded']:,}")
            lines.append(f"    Unavailable           : {r['unavailable']:,}")
            lines.append(f"    Errored               : {r['error']:,}")
            lines.append(f"    Plates attempted      : {attempted:,}")
            lines.append(f"    Avg time per plate    : {avg_time_str}")

        c = combined_by_name[name]
        c_attempted = c["downloaded"] + c["unavailable"] + c["error"]
        c_avg = (c["duration_seconds"] / c_attempted) if c_attempted > 0 else None
        c_avg_str = f"{c_avg:.3f}s" if c_avg is not None else "n/a"
        lines.append(f"  Combined (both phases):")
        lines.append(f"    Total duration        : {format_eta(c['duration_seconds'])} ({c['duration_seconds']:.2f}s)")
        lines.append(f"    Total downloaded      : {c['downloaded']:,}")
        lines.append(f"    Total unavailable     : {c['unavailable']:,}")
        lines.append(f"    Total errored         : {c['error']:,}")
        lines.append(f"    Total attempted       : {c_attempted:,}")
        lines.append(f"    Overall avg time/plate: {c_avg_str}")

        size_bytes = per_object_storage[name]
        cutout_dir_for_count = BASE_DATA_DIR / sanitize_folder_name(name) / "cutouts"
        n_files = len(list(cutout_dir_for_count.glob("*.fits"))) if cutout_dir_for_count.exists() else 0
        lines.append(f"  Storage:")
        lines.append(f"    Cutout directory size : {format_bytes(size_bytes)}")
        lines.append(f"    FITS files on disk    : {n_files:,}")

    # Final grand-total section -- Phase 1 + Phase 2 combined, all objects.
    lines.append("")
    lines.append("=" * 78)
    lines.append("FINAL TOTALS (Phase 1 + Phase 2 combined, all objects)")
    lines.append("=" * 78)
    grand_attempted = grand_downloaded + grand_unavailable + grand_error
    grand_avg = (grand_seconds / grand_attempted) if grand_attempted > 0 else None
    grand_avg_str = f"{grand_avg:.3f}s" if grand_avg is not None else "n/a"
    lines.append(f"Total plates downloaded     : {grand_downloaded:,}")
    lines.append(f"Total plates unavailable    : {grand_unavailable:,}")
    lines.append(f"Total plates errored        : {grand_error:,}")
    lines.append(f"Total plates attempted      : {grand_attempted:,}")
    lines.append(f"Sum of per-object time      : {format_eta(grand_seconds)} "
                 f"(objects run sequentially, so this should roughly match wall-clock duration)")
    lines.append(f"Overall avg time per plate  : {grand_avg_str}")
    lines.append(f"Wall-clock batch duration   : {format_eta(batch_duration)}")
    lines.append(f"Total storage used          : {format_bytes(grand_storage_bytes)}")
    lines.append("")
    lines.append("=" * 78)

    return "\n".join(lines)


def run_batch_targets(entries, panel_widgets_map):
    """entries: list of (name, ra_deg_or_None, dec_deg_or_None) tuples, as
    returned by parse_object_list_file(). Runs run_extraction_body() for
    each entry, TWICE: once with download_limit=PHASE_1_PLATE_LIMIT for
    every object (Phase 1), then once with download_limit=None (Phase 2).

    STALL DETECTION: instead of a flat timeout on the whole phase (which
    was cutting off real, slow-but-progressing downloads and reporting them
    as 0s -- the bug this fixes), each object's download thread is polled
    periodically. Its progress "heartbeat" (updated every time a plate
    completes, see run_downloads() in run_extraction_body) is checked --
    only if there's been ZERO progress for STALL_TIMEOUT_SECONDS is the
    object given up on. A slow object that keeps completing plates, however
    slowly, is never cut off.

    phase_results tracks Phase 1 and Phase 2 OUTCOMES SEPARATELY per object
    (phase_results[name][1], [name][2]), so the final report can show
    per-phase breakdowns, per-phase all-objects totals, AND a final
    Phase1+Phase2 grand total -- none of these overwrite each other."""
    global _active_download_thread

    batch_start = datetime.now()
    batch_start_mono = time.monotonic()
    phase_results = {name: {} for name, _, _ in entries}

    def run_one_phase(phase_number, phase_label, download_limit):
        if _batch_cancel_event.is_set():
            return

        for idx, (name, ra_deg, dec_deg) in enumerate(entries, start=1):
            if _batch_cancel_event.is_set():
                with batch_log_output:
                    print(f"Batch cancelled before reaching '{name}' in {phase_label}.")
                phase_results[name][phase_number] = {
                    "target_name": name, "status": "skipped (batch cancelled)",
                    "duration_seconds": 0, "downloaded": 0, "unavailable": 0, "error": 0,
                }
                continue

            batch_status_label.value = (
                f"<span style='color:#4DD0E1;'>{phase_label}: {idx}/{len(entries)} - "
                f"{'Starting' if phase_number == 1 else 'Continuing'} {name}...</span>"
            )
            with batch_log_output:
                verb = "Starting" if phase_number == 1 else "Continuing"
                coord_note = " (via coordinates)" if ra_deg is not None else ""
                print(f"\n▶ [{idx}/{len(entries)}] {phase_label}: {verb} {name}{coord_note}...")

            object_selector.value = name
            show_panel(name)

            extract_button.disabled = True
            # heartbeat is initialized here, BEFORE run_extraction_body() is
            # even called, so it's always fresh from the moment we start
            # waiting -- run_downloads() (inside run_extraction_body) will
            # keep updating result_container["heartbeat"]["last_update"]
            # every time a plate finishes.
            result_container = {"heartbeat": {"last_update": time.monotonic()}}
            thread = run_extraction_body(
                name, panel_widgets_map[name],
                result_container=result_container,
                download_limit=download_limit,
                ra_deg=ra_deg, dec_deg=dec_deg,
            )

            if thread is None:
                phase_results[name][phase_number] = {
                    "target_name": name, "status": "failed to resolve",
                    "duration_seconds": 0, "downloaded": 0, "unavailable": 0, "error": 0,
                }
                continue

            stalled = False
            while thread.is_alive():
                time.sleep(STALL_CHECK_INTERVAL_SECONDS)
                last_update = result_container.get("heartbeat", {}).get("last_update", 0)
                idle_seconds = time.monotonic() - last_update
                if idle_seconds > STALL_TIMEOUT_SECONDS:
                    stalled = True
                    break

            if stalled:
                with batch_log_output:
                    print(f"⏱ {name} has made NO download progress for over {STALL_TIMEOUT_SECONDS}s "
                          f"(genuinely stuck, not just slow) - moving on.")
                    print(f"   Cannot force-kill the thread; it may keep running in the background.")
                phase_results[name][phase_number] = {
                    "target_name": name,
                    "status": f"stalled (no progress for {STALL_TIMEOUT_SECONDS}s)",
                    "duration_seconds": idle_seconds,
                    "downloaded": 0, "unavailable": 0, "error": 0,
                }
                continue

            item_result = result_container.get("result")
            if item_result is None:
                item_result = {"target_name": name, "status": "completed (details unavailable)",
                                "duration_seconds": 0, "downloaded": 0, "unavailable": 0, "error": 0}
            phase_results[name][phase_number] = item_result

    run_one_phase(1, f"Phase 1 (up to {PHASE_1_PLATE_LIMIT} plates)", PHASE_1_PLATE_LIMIT)
    run_one_phase(2, "Phase 2 (remaining plates)", None)

    batch_end = datetime.now()
    batch_duration = time.monotonic() - batch_start_mono

    batch_report_text = build_verbose_batch_report(entries, phase_results, batch_start, batch_end, batch_duration)

    # Filename format: download_Summary_[DATE]_[TIME].txt
    batch_report_path = BASE_DATA_DIR / f"download_Summary_{batch_start.strftime('%Y%m%d_%H%M%S')}.txt"
    try:
        batch_report_path.write_text(batch_report_text, encoding="utf-8")
    except Exception as e:
        with batch_log_output:
            print(f"Could not save batch report ({e})")

    with batch_log_output:
        print("\n" + "=" * 78)
        print("BATCH COMPLETE")
        print(batch_report_text)
        print(f"\nBatch report saved to: {batch_report_path}")

    batch_status_label.value = (
        f"<span style='color:#4CAF50;'>Batch complete - {len(entries)} objects, "
        f"took {format_eta(batch_duration)}. See {batch_report_path.name}</span>"
    )
    extract_button.disabled = False
    batch_load_button.disabled = False
    batch_cancel_button.disabled = True


def on_batch_load_clicked(b):
    global _active_batch_thread

    if _active_download_thread is not None and _active_download_thread.is_alive():
        batch_status_label.value = "<span style='color:#FFA726;'>⚠️ A single extraction is currently running - wait for it to finish.</span>"
        return
    if _active_batch_thread is not None and _active_batch_thread.is_alive():
        batch_status_label.value = "<span style='color:#FFA726;'>⚠️ A batch is already running.</span>"
        return

    path_str = batch_path_input.value.strip()
    if not path_str:
        batch_status_label.value = "<span style='color:#EF5350;'>Please enter a path to a .txt file.</span>"
        return

    try:
        entries = parse_object_list_file(path_str)
    except Exception as e:
        batch_status_label.value = f"<span style='color:#EF5350;'>Could not read file: {e}</span>"
        return

    if not entries:
        batch_status_label.value = "<span style='color:#EF5350;'>File contained no object names (after removing blanks/comments).</span>"
        return

    _batch_cancel_event.clear()
    batch_load_button.disabled = True
    batch_cancel_button.disabled = False
    extract_button.disabled = True
    batch_status_label.value = f"<span style='color:#4DD0E1;'>Building dashboards for {len(entries)} objects...</span>"

    with batch_log_output:
        clear_output(wait=True)
        print(f"Loaded {len(entries)} objects from {path_str}:")
        for name, ra_deg, dec_deg in entries:
            tag = f" [coords: {ra_deg:.4f}, {dec_deg:.4f}]" if ra_deg is not None else ""
            print(f"  - {name}{tag}")
        print(f"\nBatch plan: Phase 1 gets up to {PHASE_1_PLATE_LIMIT} plates per object, "
              f"then Phase 2 finishes each object's remainder.")
        print(f"Stall detection: {STALL_TIMEOUT_SECONDS}s of zero progress "
              f"(slow-but-working downloads are never cut off, only truly stuck ones).")

    panel_widgets_map = {}
    for name, ra_deg, dec_deg in entries:
        panel_widgets_map[name] = create_panel(name)

    if entries:
        first_name = entries[0][0]
        object_selector.value = first_name
        show_panel(first_name)

    batch_status_label.value = f"<span style='color:#4DD0E1;'>Starting batch of {len(entries)} objects...</span>"

    _active_batch_thread = threading.Thread(
        target=run_batch_targets, args=(entries, panel_widgets_map), daemon=True
    )
    _active_batch_thread.start()


def on_batch_cancel_clicked(b):
    _batch_cancel_event.set()
    batch_cancel_button.disabled = True
    batch_status_label.value = "<span style='color:#FFA726;'>Cancelling after current object/phase finishes...</span>"


batch_load_button.on_click(on_batch_load_clicked)
batch_cancel_button.on_click(on_batch_cancel_clicked)

Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

Output()